In [1]:
from scraper.conllu_test import head_greater_than_id, head_smaller_than_id, has_son, conllu_files_list, obj_or_iobj, file_to_conllu,merge_evals, merge_list_evals, conllu_merger, check_conllu_for_conditions_v3, no_weak_pronoun, has_sons, next_head, prev_head, has_no_son, check_conds, feats_include, is_sub_clause_head_core_arg, nicht, is_not, is_not_any_of, has_no_sons, is_sub_clause_head, parent_is, merge_conditions_dicts, v_SO, v_OS

from scraper.analysis_main import eval_pipeline, conllu_file_stats, file_freqs_eval, file_freqs_eval_filtered_by_conds, conllu_file_freqs



In [2]:
# list of all the values in the conllu file generated w the function conllu_possible_values()
deprel_list = ['det', 'nsubj', 'amod', 'case', 'obj', 'flat', 'ROOT', 'mark', 'obl', 'compound', 'cc', 'conj', 'punct', 'appos', 'nmod', 'aux', 'ccomp', 'advmod', 'fixed', 'acl', 'nummod', 'cop', 'advcl', 'expl:pass', 'xcomp', 'csubj', 'iobj', 'parataxis', 'dep']
upos_list = ['DET', 'NOUN', 'ADJ', 'ADP', 'PROPN', 'VERB', 'SCONJ', 'NUM', 'CCONJ', 'PUNCT', 'AUX', 'SYM', 'PRON', 'ADV', 'INTJ', 'SPACE', 'PART']
upos_list_args= ['DET', 'NOUN', 'ADJ', 'PROPN','PRON', 'ADV','NUM', 'SYM']

#EVAL 1
# SVO in simple clauses. (sense complements verbals, sense que el verb sigui una subordinada -> pot ser coordinat)
#  we allow advcl, etc to be tere though
# 1 AQUESTA ES LA CONDICIO D UN BON VERB (sense complements verbals, sense ser xcomp d un altre verb)
# no ser el main verb d una subordinada vol dir restringir molts deprels:
# aux is also out bc it marks situations such as "cal marxar", "estan a punt d entrar"
# fixed is also aut bc it is basically "ès a dir"
# compound is also out bc it classifies constructions where verbs are very close to another "vol dir", "es fa dir"
# Evitar coordinacio amb subordinades -> per evitar frases com (1, '2025-10-10 00:33:22_habitatge_65_031_011')] de a1KI_corpus_65_habitatge_gemm.conllu
# no hi ha una fill SCONJ
eval_v= {"verb": [{"upos": "VERB"}]}
conds_v_adequat = [{"upos": "VERB", is_not_any_of: [{"lemma":"ser"},{"deprel": is_sub_clause_head}, {"deprel": "aux"}, {"deprel": "fixed"}, {"deprel": "compound"}, {"deprel": "conj", parent_is : {"upos":"VERB", "deprel" : is_sub_clause_head} }], has_no_sons: [{"deprel": is_sub_clause_head_core_arg}, {"deprel":"cop"}, {feats_include: {"PronType": "Rel"}}, {"upos": "SCONJ"}]  }]
eval_v_adequat_a = {"verb_ok": conds_v_adequat }



# eliminar SVS i OVO del v adequat

conds_not_OVO =  [{"upos": "VERB", nicht(has_sons): [{"deprel": obj_or_iobj, "head": head_smaller_than_id,"lemma": no_weak_pronoun, is_not_any_of: [{feats_include: {"PronType": "Rel"}}, {"upos": "ADP"}, {"upos":"ADJ"}, {"upos":"ADV"}], has_no_sons: [{"upos": "ADP"}, {"deprel": "case"}, {"deprel": "cop"}, {"deprel": "mark"}  ]}, {"deprel": "obj", "head": head_greater_than_id,"lemma": no_weak_pronoun, is_not_any_of: [{feats_include: {"PronType": "Rel"}}, {"upos": "ADP"}, {"upos":"ADJ"}, {"upos":"ADV"}], has_no_sons: [{"upos": "ADP"}, {"deprel": "case"}, {"deprel": "cop"}, {"deprel": "mark"}  ]} ]}]
eval_not_OVO = {"not_OVO": conds_not_OVO}

# eliminar SVS del v adequat
eval_not_SVS= {"not_SVS":  [{"upos": "VERB", nicht(has_sons): [{"deprel": "nsubj", "lemma": no_weak_pronoun, is_not: {feats_include: {"PronType": "Rel"}}, "head": head_smaller_than_id, has_no_son: {"deprel": "cop"} }, {"deprel": "nsubj", "lemma": no_weak_pronoun, is_not: {feats_include: {"PronType": "Rel"}}, "head": head_greater_than_id, has_no_son: {"deprel": "cop"} }] }]}

eval_v_adequat = {"v-ok":merge_list_evals([eval_v_adequat_a, eval_not_SVS, eval_not_OVO])["verb_ok__UND__verb_ok__UND__not_SVS__UND__not_OVO"]}



# EVAL 1.1 checking the deprels of the simple verb to make sure everything is ok
eval_verb_deprels = {f"verb_{deprel}": [{"upos": "VERB", "deprel": deprel}]   for deprel in deprel_list}
eval_verb_derpel_no_subclause_arg = {f"verb_no_subclause_args_{deprel}": [{"upos": "VERB", "deprel": deprel, nicht(has_sons): [{"deprel": is_sub_clause_head_core_arg}]}]   for deprel in deprel_list}

# Defining the object of a verb (clause type independent)
# eliminem pronoms febles, eliminem complements preposicionals, eliminem pronoms relatius amb funcio d objecte,
# eliminem tambe deprel case per eliminar coses com ara: "com a mesures s han aplicat.."
# eliminem tambe que l obj pugui tenir upos ADP per eliminar "donar per tancat, etc"
# possible case study: cleft sentences
conds_v_obj_good= [{"upos": "VERB", has_sons: [{"deprel": "obj", "lemma": no_weak_pronoun, is_not_any_of: [{feats_include: {"PronType": "Rel"}}, {"upos":"ADP"}, {"upos":"ADJ"}, {"upos":"ADV"}], has_no_sons: [{"upos": "ADP"}, {"deprel": "case"}, {"deprel": "cop"}, {"deprel": "mark"}] }] }]
conds_v_NOT_obj_good = [{"upos": "VERB", is_not_any_of: [{has_sons: [{"deprel": "obj", "lemma": no_weak_pronoun, is_not_any_of: [{feats_include: {"PronType": "Rel"}}, {"upos":"ADP"}, {"upos":"ADJ"}, {"upos":"ADV"}], has_no_sons: [{"upos": "ADP"}, {"deprel": "case"}, {"deprel": "cop"},  {"deprel": "mark"}]}] }]}]
eval_obj = {"v_w_obj": conds_v_obj_good, "v_w_no_obj": conds_v_NOT_obj_good}

#iobj_good eval
conds_v_iobj_good= [{"upos": "VERB", has_sons: [{"deprel": "iobj", "lemma": no_weak_pronoun, is_not_any_of: [{feats_include: {"PronType": "Rel"}}, {"upos":"ADP"}, {"upos":"ADJ"}, {"upos":"ADV"}, {"upos":"AUX"}], has_no_sons: [{"upos": "ADP"}, {"deprel": "case"}, {"deprel": "cop"}, {"deprel": "mark"}] }] }]
conds_v_NOT_iobj_good = [{"upos": "VERB", is_not_any_of: [{has_sons: [{"deprel": "iobj", "lemma": no_weak_pronoun, is_not_any_of: [{feats_include: {"PronType": "Rel"}}, {"upos":"ADP"}, {"upos":"ADJ"}, {"upos":"ADV"}], has_no_sons: [{"upos": "ADP"}, {"deprel": "case"}, {"deprel": "cop"},  {"deprel": "mark"}]}] }]}]
eval_iobj = {"v_w_iobj": conds_v_iobj_good, "v_w_no_iobj": conds_v_NOT_iobj_good}
list_evals_iobj= [eval_v_adequat, merge_evals(eval_v_adequat, eval_iobj), merge_evals(eval_v_adequat, merge_evals(eval_obj, eval_iobj)) ]


# todo: add iobj to obj, change name to v2
# obj_better! added iobj
conds_v_obj_good2= [{"upos": "VERB", has_sons: [    {"deprel": obj_or_iobj, "lemma": no_weak_pronoun, is_not_any_of: [{feats_include: {"PronType": "Rel"}}, {"upos":"ADP"}, {"upos":"ADJ"}, {"upos":"ADV"}], has_no_sons: [{"upos": "ADP"}, {"deprel": "case"}, {"deprel": "cop"}, {"deprel": "mark"}] }      ] }]
conds_v_NOT_obj_good2 = [{"upos": "VERB", is_not_any_of: [{has_sons: [ {"deprel": obj_or_iobj, "lemma": no_weak_pronoun, is_not_any_of: [{feats_include: {"PronType": "Rel"}}, {"upos":"ADP"}, {"upos":"ADJ"}, {"upos":"ADV"}, {"upos":"AUX"}], has_no_sons: [{"upos": "ADP"}, {"deprel": "case"}, {"deprel": "cop"}, {"deprel": "mark"}] }] }]}]
# using second version of obj which includes iobj as obj
eval_obj2 = {"v_obj2": conds_v_obj_good2, "v_no_obj2": conds_v_NOT_obj_good2}

conds_v_obj_good_satanic= [conds_v_NOT_iobj_good[0]]
eval_obj=eval_obj2


#"v_w_obj_und_no_obj": merge_conditions_dicts(conds_v_obj_good[0], conds_v_NOT_obj_good[0])

# Defining a proper subject of a verb (clause type independent)
# i
conds_verb_wsubj_true = [{"upos": "VERB", has_sons: [{"deprel": "nsubj", "lemma": no_weak_pronoun, is_not: {feats_include: {"PronType": "Rel"}}, has_no_son: {"deprel": "cop"} }] }]
conds_v_NOT_subj = [{"upos": "VERB", nicht(has_sons): [{"deprel": "nsubj", "lemma": no_weak_pronoun, is_not: {feats_include: {"PronType": "Rel"}},has_no_son: {"deprel": "cop"} }] }]
eval_subj = {"v_subj": conds_verb_wsubj_true, "v_no_subj": conds_v_NOT_subj }
# "v_w_subj_und_no_subj": merge_conditions_dicts(conds_verb_wsubj_true[0], conds_v_NOT_subj[0])


# LIST for EVAL 1:
# this is the list of evaluations for the simple presence of S V O in "simple" clauses, this is, verbal predicates with no clausal core arguments which are also not the main verb of a subordinate clause
list_evals_presence_SVO = [eval_v, eval_v_adequat, merge_evals(eval_v_adequat, eval_subj), merge_evals(eval_v_adequat, eval_obj), merge_evals(eval_v_adequat, merge_evals(eval_subj, eval_obj)), ]


# EVAL 1.2
# what is the upos class ob our beautiful verb arguments?
# obj is not updated for upos restrictions in "good" objects (no adj, no adp)
eval_obj_upos= {f"verb__obj_{upos}": [{"upos": "VERB", has_sons: [{"deprel": "obj", "upos": upos ,"lemma": no_weak_pronoun, is_not: {feats_include: {"PronType": "Rel"}}, has_no_sons: [{"upos": "ADP"}, {"deprel": "case"}, {"deprel": "cop"}, {"deprel": "mark"}]}]}] for upos in upos_list_args}
eval_subj_upos= {f"verb_subj_{upos}": [{"upos": "VERB", has_sons: [{"deprel": "nsubj", "lemma": no_weak_pronoun, "upos":upos,  is_not: {feats_include: {"PronType": "Rel"}},has_no_son: {"deprel": "cop"} }] }] for upos in upos_list_args}
list_evals_SVO_type= [eval_v, eval_obj, eval_subj, eval_subj_upos,eval_obj_upos]


#EVAL 2_
# order of the verb arguments in EVAL 1
# -> successful
# order of the object: OV vs VO
conds_v_obj_good_OV = [{"upos": "VERB", has_sons: [{"deprel": obj_or_iobj, "head": head_greater_than_id,"lemma": no_weak_pronoun, is_not_any_of: [{feats_include: {"PronType": "Rel"}}, {"upos": "ADP"}, {"upos":"ADJ"}, {"upos":"ADV"}], has_no_sons: [{"upos": "ADP"}, {"deprel": "case"}, {"deprel": "cop"},  {"deprel": "mark"}]}]}]
conds_v_obj_good_VO = [{"upos": "VERB", has_sons: [{"deprel": obj_or_iobj, "head": head_smaller_than_id,"lemma": no_weak_pronoun, is_not_any_of: [{feats_include: {"PronType": "Rel"}}, {"upos": "ADP"}, {"upos":"ADJ"}, {"upos":"ADV"}], has_no_sons: [{"upos": "ADP"}, {"deprel": "case"}, {"deprel": "cop"}, {"deprel": "mark"}]}]}]
conds_v_obj_good_OVO = [{"upos": "VERB", has_sons: [{"deprel": obj_or_iobj, "head": head_smaller_than_id,"lemma": no_weak_pronoun, is_not_any_of: [{feats_include: {"PronType": "Rel"}}, {"upos": "ADP"}, {"upos":"ADJ"}, {"upos":"ADV"}], has_no_sons: [{"upos": "ADP"}, {"deprel": "case"}, {"deprel": "cop"}, {"deprel": "mark"}  ]}, {"deprel": "obj", "head": head_greater_than_id,"lemma": no_weak_pronoun, is_not_any_of: [{feats_include: {"PronType": "Rel"}}, {"upos": "ADP"}, {"upos":"ADJ"}, {"upos":"ADV"}], has_no_sons: [{"upos": "ADP"}, {"deprel": "case"}, {"deprel": "cop"}, {"deprel": "mark"}  ]} ]}]

eval_OVO = {"obj_OV": conds_v_obj_good_OV, "obj_VO": conds_v_obj_good_VO, "obj_OVO": conds_v_obj_good_OVO}



# order of the subject
conds_v_subj_good_SV = [{"upos": "VERB", has_sons: [{"deprel": "nsubj", "lemma": no_weak_pronoun, is_not: {feats_include: {"PronType": "Rel"}}, "head": head_greater_than_id, has_no_son: {"deprel": "cop"} }] }]
conds_v_subj_good_VS = [{"upos": "VERB", has_sons: [{"deprel": "nsubj", "lemma": no_weak_pronoun, is_not: {feats_include: {"PronType": "Rel"}}, "head": head_smaller_than_id, has_no_son: {"deprel": "cop"} }] }]
conds_v_subj_good_SVS = [{"upos": "VERB", has_sons: [{"deprel": "nsubj", "lemma": no_weak_pronoun, is_not: {feats_include: {"PronType": "Rel"}}, "head": head_smaller_than_id, has_no_son: {"deprel": "cop"} }, {"deprel": "nsubj", "lemma": no_weak_pronoun, is_not: {feats_include: {"PronType": "Rel"}}, "head": head_greater_than_id, has_no_son: {"deprel": "cop"} }] }]

eval_SVS= {"subj_SV": conds_v_subj_good_SV, "subj_VS": conds_v_subj_good_VS, "subj_SVS": conds_v_subj_good_SVS}

list_evals_SVSOVO_ausschliessen = [eval_v_adequat, merge_evals(eval_OVO, eval_v_adequat), merge_evals(eval_SVS, eval_v_adequat), eval_v_adequat_a, merge_evals(eval_OVO, eval_v_adequat_a), merge_evals(eval_SVS, eval_v_adequat_a)]



# final test: SOV vs OSV, VSO vs VOS
conds_SO = [{"head": v_SO, "lemma": no_weak_pronoun}]
conds_OS = [{"head": v_OS}]

eval_SO_order = {"SO": conds_SO, "OS": conds_OS}



list_eval_SVO_order= list_evals_presence_SVO + [merge_evals(eval_v_adequat ,eval_OVO)] + [merge_evals(eval_v_adequat ,eval_SVS)] + [merge_evals(eval_v_adequat,merge_evals(merge_evals(eval_OVO, eval_SO_order), eval_SVS))]


# EVAL 3
# same but in subordinate clauses
# TODO: clarify if the distinction between sub itself and coordinated with a subordinate clause makes sense or not
# CONDICIONS PER UNA SUBORDINADA DE FUNCIO X
# restringim a les que tenen un nexe de tipus SCONJ, q te deprel mark -> que no fa cap funcio
list_sub_deprels = ["csubj", "ccomp", "advcl", "acl", "xcomp"]

eval_v_sub ={}
list_eval_SVO_presence ={}
for deprel in list_sub_deprels:
    eval_v_sub[deprel] = {f"v_sub_deprel_{deprel}": [{"upos": "VERB", "deprel": deprel, has_son: {"upos": "SCONJ", "deprel":"mark"}, nicht(has_sons): [{"deprel": is_sub_clause_head_core_arg}] }]}
    list_eval_SVO_presence[deprel] =[eval_v, eval_v_sub[deprel], merge_evals(eval_v_sub[deprel], eval_subj), merge_evals(eval_v_sub[deprel], eval_obj), merge_evals(eval_v_sub[deprel], merge_evals(eval_subj, eval_obj)), ]
eval_v_sub_types = {f"v_sub_deprel_{deprel}": [{"upos": "VERB", "deprel": deprel, has_son: {"upos": "SCONJ", "deprel":"mark"}, nicht(has_sons): [{"deprel": is_sub_clause_head_core_arg}] }] for deprel in list_sub_deprels}

eval_v_sub_types_coord = {f"v_sub_coord_deprel_{deprel}": [{"upos": "VERB", "deprel": "conj", parent_is: {"deprel": deprel},has_son: {"upos": "SCONJ", "deprel":"mark"}, nicht(has_sons): [{"deprel": is_sub_clause_head_core_arg}] }] for deprel in list_sub_deprels}

# EVAL SVO presence in subordinates and coordinates with subordinates
# DONT USE THIS ONE IT TAKES 35 min
list_evals_SVO_presence_subordinates = [eval_v, eval_v_sub_types, merge_evals(eval_v_sub_types, eval_subj), merge_evals(eval_v_sub_types, eval_obj), merge_evals(eval_v_sub_types, merge_evals(eval_subj, eval_obj)), ]
list_evals_SVO_presence_sub_coord = [eval_v, eval_v_sub_types_coord, merge_evals(eval_v_sub_types_coord, eval_subj), merge_evals(eval_v_sub_types_coord, eval_obj), merge_evals(eval_v_sub_types_coord, merge_evals(eval_subj, eval_obj)), ]

list_evals_SVO_presence_sub_both = list_evals_SVO_presence_subordinates + list_evals_SVO_presence_sub_coord

#EVAL SVO order in subordinates and coordinates w subordinates
list_eval_SVO_order_subordinates = [eval_v, eval_v_sub_types] + [merge_evals(eval_v_sub_types ,eval_OVO)] + [merge_evals(eval_v_sub_types ,eval_SVS)] + [merge_evals(eval_v_sub_types, merge_evals(merge_evals(eval_OVO, eval_SO_order), eval_SVS))]
list_eval_SVO_order_sub_coord = list_evals_SVO_presence_sub_coord + [merge_evals(eval_v_sub_types_coord ,eval_OVO)] + [merge_evals(eval_v_sub_types_coord ,eval_SVS)] + [merge_evals(eval_v_sub_types_coord, merge_evals(eval_OVO, eval_SVS))]

list_evals_SVO_order_sub_both = list_eval_SVO_order_subordinates + list_eval_SVO_order_sub_coord

# EVAL 3.5
# tipus de subordinades mes comprimit
# en comptes de tenir un nexe "SCONJ" (que, si,) que passa amb les q tenen nexe "ADP" (cansats de compartir, per destituir el president..)
# i a mes el verb esta en infinitiu:
eval_v_sub_types3 = {f"v_sub2_infinitiu_deprel_{deprel}": [{"upos": "VERB", feats_include: {'VerbForm':'Inf'}, "deprel": deprel, has_son: {"upos": "ADP", "deprel":"mark"}, nicht(has_sons): [{"deprel": is_sub_clause_head_core_arg}] }] for deprel in list_sub_deprels}
eval_v_sub_types3_coord = {f"v_sub_infinitiu_coord_deprel_{deprel}": [{"upos": "VERB", feats_include: {'VerbForm':'Inf'}, "deprel": "conj", parent_is: {"deprel": deprel},has_son: {"upos": "ADP", "deprel":"mark"}, nicht(has_sons): [{"deprel": is_sub_clause_head_core_arg}] }] for deprel in list_sub_deprels}
list_evals_subs_SVO_presence = [eval_v_sub_types3, eval_v_sub_types3_coord, merge_evals(eval_v_sub_types3, eval_obj), merge_evals(eval_v_sub_types3, eval_subj)]


conds_v_obj_OV_adj = [{"upos": "VERB", has_sons: [{"deprel": "obj", "upos":"ADJ","head": head_greater_than_id,"lemma": no_weak_pronoun, is_not: {feats_include: {"PronType": "Rel"}}, has_no_sons: [{"upos": "ADP"}, {"deprel": "case"}, {"deprel": "mark"}]}]}]
conds_v_obj_VO_adj = [{"upos": "VERB", has_sons: [{"deprel": "obj", "upos":"ADJ", "head": head_smaller_than_id,"lemma": no_weak_pronoun, is_not: {feats_include: {"PronType": "Rel"}}, has_no_sons: [{"upos": "ADP"}, {"deprel": "case"}, {"deprel": "mark"}]}]}]


eval_obj_adj_problem = {"OV_adj": conds_v_obj_OV_adj , "VO_not_adj": conds_v_obj_VO_adj}
list_OVO_adj_problem = [merge_evals(eval_v_adequat, eval_obj_adj_problem)]


# EVAL 4
# NOUN PHRASE WHATS UP
# idea: upos based
# nur nominale sintagmas die nur nomen haben

nom_deprels = ["nmod","appos", "nummod","acl", "amod", "det", "clf", "case"]
#general noun conditions: maybe i add more
conds_noun_ok= [{"upos":"NOUN"}]
eval_noun= {"eval_n": conds_noun_ok}


# ADJ + N order
conds_adj_n = [{"upos": "NOUN", has_sons: [{"upos": "ADJ", "deprel":"amod", "head": head_greater_than_id, is_not_any_of: [{"deprel": "cop"}, {"deprel": "acl"}]}]}]
conds_n_adj =[{"upos": "NOUN", has_sons: [{"upos": "ADJ", "deprel":"amod", "head": head_smaller_than_id, is_not_any_of: [{"deprel": "cop"}, {"deprel": "acl"}]}]}]
conds_adj_n_adj= merge_conditions_dicts(conds_n_adj[0], conds_adj_n[0])

eval_adj_n = {"adj_n": conds_adj_n, "n_adj":conds_n_adj, "adj_n_adj":[conds_adj_n_adj]}

# ADV + ADJ
conds_adv_adj_n = [{"upos": "NOUN", has_sons: [{"upos": "ADJ", "head": head_greater_than_id, "deprel": "amod",  has_sons: [{"upos": "ADV", "deprel":"advmod"}] }]}]
conds_n_adv_adj =[{"upos": "NOUN", has_sons: [{"upos": "ADJ", "head": head_smaller_than_id, "deprel": "amod", has_sons: [{"upos": "ADV", "deprel":"advmod"}] }]}]
conds_adv_adj_n_adj= [merge_conditions_dicts(conds_n_adv_adj[0], conds_adv_adj_n[0])]

eval_adv_adj_n = {"adv_adj_n": conds_adv_adj_n, "n_adv_adj": conds_n_adv_adj, "n_adv_adj_beide":conds_adv_adj_n_adj}


list_eval_adjs= [eval_noun, eval_adj_n, eval_adv_adj_n]

# EVAL 5:
# numerals
conds_n_num= [{"upos": "NOUN", has_sons: [{"deprel": "nummod"}]}]
conds_n_num_upos= [{"upos": "NOUN", has_sons: [{"upos": "NUM"}]}]
conds_n_num2 = [{"upos":"NOUN", has_sons: [{"upos": "NUM", is_not: {"deprel": "nummod"} } ] }]
conds_n_num3 = [{"upos":"NOUN", has_sons: [{"deprel": "nummod", is_not: {"upos": "NUM"} } ] }]

eval_n_num ={"n_nummod": conds_n_num, "n_num_upos": conds_n_num_upos, "num_not_nummod": conds_n_num2, "nummod_not_num": conds_n_num3 }

list_nom_evals= [eval_noun, eval_adj_n, eval_adv_adj_n]


# EVAL 6 : feats dels adverbis q acompanyen adjectius
conds_adv_adj= [{"upos": "ADV", parent_is: {"upos": "ADJ", "head": head_greater_than_id, parent_is: {"upos": "NOUN"}}}]
conds_n_adv_adj= [{"upos": "ADV", parent_is: {"upos": "ADJ", "head": head_smaller_than_id, parent_is: {"upos": "NOUN"}}}]

eval_adv_adj = {"adv_adj_n": conds_adv_adj, "n_adv_adj": conds_n_adv_adj}
adv_possible_feats= [{'Polarity': 'Neg'},{'Degree': 'Cmp'}]

eval_adv_types= {f"feat{feature}": [{feats_include: feature}] for feature in adv_possible_feats}
llista_adv_types = [eval_adv_adj, merge_evals(eval_adv_types, eval_adv_adj)]

# EVAL 7: noun templates ;)
# pos 0 determiners
conds_l0={}
conds_l0[0]= [{has_no_sons: [{"upos": "DET", "deprel":"det", "head":head_greater_than_id} ]}]
conds_l0[1]= [{has_sons: [{"upos": "DET", "deprel":"det", "head":head_greater_than_id, has_no_son: {"deprel": "det"}} ], has_no_sons: [{"upos": "DET", "deprel":"det", "head":head_greater_than_id, has_son: {"deprel": "det", has_no_son: {"deprel": "det"}} }] }]

conds_l0[2]= [{has_sons: [{"upos": "DET", "deprel":"det", "head":head_greater_than_id, has_son: {"deprel": "det", has_no_son: {"deprel": "det"}} } ], has_no_sons: [{"upos": "DET", "deprel":"det", "head":head_greater_than_id, has_no_son: {"deprel": "det"}} ] }]

conds_l0[3]= [{has_sons: [{"upos": "DET", "deprel":"det", "head":head_greater_than_id, has_son: {"deprel": "det", has_no_son: {"deprel": "det"}} }, {"upos": "DET", "deprel":"det", "head":head_greater_than_id, has_no_son: {"deprel": "det"}} ]}]


eval_det0 = {"det0_"+str(k): conds_l0[k] for k in conds_l0.keys()}

# pos 1 adjectives
conds_l1={}
conds_l1[0] = [{has_no_sons: [{"upos": "ADJ", "deprel":"amod", "head":head_greater_than_id} ]}]
conds_l1[1]= [{has_sons: [{"upos": "ADJ", "deprel":"amod", "head":head_greater_than_id, has_no_sons: [{"upos": "ADV"}]} ]}]
conds_l1[2]= [{has_sons: [{"upos": "ADJ", "deprel":"amod", "head":head_greater_than_id, has_sons: [{"upos": "ADV"}]} ]}]

eval_adj1= {"adj1_"+ str(k): conds_l1[k] for k in conds_l1.keys()}

# pos 2 : the head noun
conds_nom=[{"upos": "NOUN", has_no_sons: [{"upos": "ADP"}, {"deprel":"cop"}]}]
eval_nom= {"nom": conds_nom}

# pos 3: post nom det (incl separar pronoms mal reconeguts: ex: "una evacuacio com la de", )
conds_l3={}
conds_l3[0]= [{has_no_sons: [{"upos": "DET", "deprel":"det", "head":head_smaller_than_id, has_no_sons: [{"deprel": "mark"}, {"deprel":"appos"}, {"deprel":"det"}] } ]}]
conds_l3[1]= [{has_sons: [{"upos": "DET", "deprel":"det", "head":head_smaller_than_id, has_no_sons: [{"deprel": "mark"}, {"deprel":"appos"}, {"deprel":"det"}] } ]}]
#conds_l3[2]= [{has_sons: [{"upos": "DET", "deprel":"det", "head":head_smaller_than_id, has_son: {"deprel": "mark"}, has_no_sons: [ {"deprel":"appos"}, {"deprel":"det"}] } ]}]
#conds_l3[3]= [{has_sons: [{"upos": "DET", "deprel":"det", "head":head_smaller_than_id, has_son: {"deprel": "appos"}, has_no_sons: [ {"deprel":"mark"}, {"deprel":"det"}] } ]}]

eval_det3 = {"det3_"+str(k): conds_l3[k] for k in conds_l3.keys()}

# pos 4 post nom adj
# incloent la coma

conds_l4={}
conds_l4[0] = [{has_no_sons: [{"upos": "ADJ", "deprel":"amod", "head":head_smaller_than_id} ]}]
conds_l4[1]= [{has_sons: [{"upos": "ADJ", "deprel":"amod", "head":head_smaller_than_id, has_no_sons: [{"upos": "ADV"}, {"upos":"PUNCT"}]} ]}]
#conds_l4[1]= [{has_sons: [{"upos": "ADJ", "deprel":"amod", "head":head_smaller_than_id, has_no_sons: [{"upos": "ADV"}]} ]}]
conds_l4[2]= [{has_sons: [{"upos": "ADJ", "deprel":"amod", "head":head_smaller_than_id, has_sons: [{"upos": "ADV"}], has_no_sons: [{"upos":"PUNCT"}]} ]}]
#conds_l4[3]= [{has_sons: [{"upos": "ADJ", "deprel":"amod", "head":head_smaller_than_id, has_sons: [{"upos":"PUNCT"}],has_no_sons: [{"upos": "ADV"}]} ]}]
#conds_l4[4]= [{has_sons: [{"upos": "ADJ", "deprel":"amod", "head":head_smaller_than_id, has_sons: [{"upos":"PUNCT"}, {"upos": "ADV"}]} ]}]

eval_adj4= {"adj4_a"+ str(k): conds_l4[k] for k in conds_l4.keys()}

# pos 5 numerals
conds_l5={}
conds_l5[0]= [{has_no_sons: [{"upos": "NUM", "deprel":"nummod"} ]}]
conds_l5[1]= [{has_sons: [{"upos": "NUM", "deprel":"nummod"} ]}]

eval_num= {"num"+ str(k): conds_l5[k] for k in conds_l5.keys()}

# pos 6 appos

conds_l6 = {}
conds_l6[0]= [{has_no_sons: [{"deprel":"appos"} ]}]
conds_l6[1]= [{has_sons: [{"deprel":"appos"} ]}]

eval_appos = {"appos"+ str(k): conds_l6[k] for k in conds_l6.keys()}

list_n_template= [eval_nom, eval_det0, eval_adj1, eval_det3, eval_adj4, eval_num, eval_appos]
list_n_template_all= [merge_evals(eval_nom, item) for item in list_n_template]



list_template_combined= [eval_nom, merge_list_evals([eval_nom]+list_n_template)]
print(len(list_template_combined[1]))

288


In [3]:
codi = "b14"

 #corpus
tv3_corpus=f"{codi}_tv3_corpus.conllu"
KI_corpus=f"{codi}_KI_corpus.conllu"

tv3_small= f"b6tv3_corpus_7_ucraina.conllu"
KI_small= f"b2KI_corpus_14_ucraina_gemm.conllu"

tv3_mid= "b14tv3_corpus_121_ucraina.conllu"
KI_mid= "b14KI_corpus_117_ucraina_gemma-3-27b-it.conllu"

sample= ["b14tv3_corpus_178_migracions.conllu"]
mini = ["mini.conllu"]
#some handy corpus lists:
file_list=[tv3_corpus, KI_corpus]
small_file_list=[tv3_small, KI_small]
mid_file_list=[tv3_mid, KI_mid]

In [5]:
mega_SVO_eval_list = list_evals_SVO_order_sub_both+list_eval_SVO_order
list_evals_SVO_order_sub_both = list_eval_SVO_order_subordinates + list_eval_SVO_order_sub_coord
# SVO order tkaes 5 min -> 7 min, 20min mit der 200 liste


eval_pipeline(list_template_combined, sample, f"{codi}_SN_sample_err_check.csv", basic_run=True, extra_eval_files=True)
# big_SN_combined.csv basic true ist already gelaufen
# deures: 
#eval_pipeline(list_eval_SVO_order_subordinates, file_list, f"{codi}_big_SVO_order_sub.csv")
#eval_pipeline(list_evals_SVO_order_sub_both, file_list, f"{codi}_big_SVO_order_sub_i_coord.csv")

Evaluating list 1 of 2 on b14tv3_corpus_178_migracions.conllu
	Evaluating nom on b14tv3_corpus_178_migracions.conllu
	 	Evaluated nom on b14tv3_corpus_178_migracions.conllu, it took 1.3675620555877686
done with list, it took 1.367663860321045
Evaluating list 2 of 2 on b14tv3_corpus_178_migracions.conllu
	Evaluating nom__UND__nom__UND__nom__UND__det0_0__UND__adj1_0__UND__det3_0__UND__adj4_a0__UND__num0__UND__appos0 on b14tv3_corpus_178_migracions.conllu
	 	Evaluated nom__UND__nom__UND__nom__UND__det0_0__UND__adj1_0__UND__det3_0__UND__adj4_a0__UND__num0__UND__appos0 on b14tv3_corpus_178_migracions.conllu, it took 10.570029497146606
	Evaluating nom__UND__nom__UND__nom__UND__det0_0__UND__adj1_0__UND__det3_0__UND__adj4_a0__UND__num0__UND__appos1 on b14tv3_corpus_178_migracions.conllu
	 	Evaluated nom__UND__nom__UND__nom__UND__det0_0__UND__adj1_0__UND__det3_0__UND__adj4_a0__UND__num0__UND__appos1 on b14tv3_corpus_178_migracions.conllu, it took 8.627066612243652
	Evaluating nom__UND__nom__UND

0

adding ucraina
adding franca
adding eleccions
adding feina
adding xina
adding erc
adding russia
adding meteorologia
adding salut
adding israel
adding educacio
adding musica
adding estatsunits
adding incendis
adding psoe
adding habitatge
adding migracions
['b14tv3_corpus_121_ucraina.conllu', 'b14tv3_corpus_177_franca.conllu', 'b14tv3_corpus_139_eleccions.conllu', 'b14tv3_corpus_174_feina.conllu', 'b14tv3_corpus_171_xina.conllu', 'b14tv3_corpus_158_erc.conllu', 'b14tv3_corpus_163_russia.conllu', 'b14tv3_corpus_122_meteorologia.conllu', 'b14tv3_corpus_178_salut.conllu', 'b14tv3_corpus_105_israel.conllu', 'b14tv3_corpus_166_educacio.conllu', 'b14tv3_corpus_143_musica.conllu', 'b14tv3_corpus_167_estatsunits.conllu', 'b14tv3_corpus_136_incendis.conllu', 'b14tv3_corpus_167_psoe.conllu', 'b14tv3_corpus_159_habitatge.conllu', 'b14tv3_corpus_178_migracions.conllu']


In [5]:
print(eval_v_adequat)

{'verb_ok__UND__verb_ok__UND__not_SVS__UND__not_OVO': [{'upos': 'VERB', <function is_not_any_of at 0x7f1ddae71f80>: [{'lemma': 'ser'}, {'deprel': <function is_sub_clause_head at 0x7f1ddae728e0>}, {'deprel': 'aux'}, {'deprel': 'fixed'}, {'deprel': 'compound'}, {'deprel': 'conj', <function parent_is at 0x7f1ddae71e40>: {'upos': 'VERB', 'deprel': <function is_sub_clause_head at 0x7f1ddae728e0>}}, {'lemma': 'ser'}, {'deprel': <function is_sub_clause_head at 0x7f1ddae728e0>}, {'deprel': 'aux'}, {'deprel': 'fixed'}, {'deprel': 'compound'}, {'deprel': 'conj', <function parent_is at 0x7f1ddae71e40>: {'upos': 'VERB', 'deprel': <function is_sub_clause_head at 0x7f1ddae728e0>}}], <function has_no_sons at 0x7f1ddae72340>: [{'deprel': <function is_sub_clause_head_core_arg at 0x7f1ddae72840>}, {'deprel': 'cop'}, {<function feats_include at 0x7f1ddae71da0>: {'PronType': 'Rel'}}, {'upos': 'SCONJ'}, {'deprel': <function is_sub_clause_head_core_arg at 0x7f1ddae72840>}, {'deprel': 'cop'}, {<function feat

In [12]:
# codeblock to merge all files into one fat file :)

for item in list_evals_iobj:
    print(item.keys())

dict_keys(['verb_ok'])
dict_keys(['verb_ok__UND__v_w_iobj', 'verb_ok__UND__v_w_no_iobj'])
dict_keys(['verb_ok__UND__v_obj2__UND__v_w_iobj', 'verb_ok__UND__v_obj2__UND__v_w_no_iobj', 'verb_ok__UND__v_no_obj2__UND__v_w_iobj', 'verb_ok__UND__v_no_obj2__UND__v_w_no_iobj'])


In [4]:

import json
import os

llista_temes={"ucraina": 170,"franca": 97,"eleccions":44,"feina": 72,"xina":61,"erc": 125,  "russia": 147,"meteorologia":1280,"salut":550, "israel": 112,  "educacio":269, "musica":234, "estatsunits": 306, "incendis":117,  "psoe":110, "habitatge":125, "migracions":136}
llista=llista_temes.keys()
tipus= "KI"
codi="b14"
listy= conllu_files_list(llista, codi, tipus)
print(listy)
liste= []
for file in listy:
    d=conllu_file_stats(file)
    d["filename"]= file
    liste.append(d)



os.makedirs("conlludata", exist_ok=True)
with open("conlludata/sample.json", "w+") as f:
    json.dump(liste, f)



adding ucraina
adding franca
adding eleccions
adding feina
adding xina
adding erc
adding russia
adding meteorologia
adding salut
adding israel
adding educacio
adding musica
adding estatsunits
adding incendis
adding psoe
adding habitatge
adding migracions
['b14KI_corpus_117_ucraina_gemma-3-27b-it.conllu', 'b14KI_corpus_177_franca_gemma-3-27b-it.conllu', 'b14KI_corpus_139_eleccions_gemma-3-27b-it.conllu', 'b14KI_corpus_174_feina_gemma-3-27b-it.conllu', 'b14KI_corpus_171_xina_gemma-3-27b-it.conllu', 'b14KI_corpus_158_erc_gemma-3-27b-it.conllu', 'b14KI_corpus_163_russia_gemma-3-27b-it.conllu', 'b14KI_corpus_122_meteorologia_gemma-3-27b-it.conllu', 'b14KI_corpus_178_salut_gemma-3-27b-it.conllu', 'b14KI_corpus_105_israel_gemma-3-27b-it.conllu', 'b14KI_corpus_166_educacio_gemma-3-27b-it.conllu', 'b14KI_corpus_143_musica_gemma-3-27b-it.conllu', 'b14KI_corpus_167_estatsunits_gemma-3-27b-it.conllu', 'b14KI_corpus_136_incendis_gemma-3-27b-it.conllu', 'b14KI_corpus_167_psoe_gemma-3-27b-it.conllu',

dict_keys(['sent_id', 'text', 'article_id', 'batch_id', 'custom_sent_id', 'filename', 'url'])
dict_keys(['id', 'form', 'lemma', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc'])


{'words': 771980,
 'sentences': 29116,
 'av. sentence length': 26.51394422310757,
 'articles': 2460,
 'sentences/article': 11.835772357723577,
 'av. tree height': 4.874021156752301,
 'max tree height': 15,
 'tree height / sent length': 0.20375196790432715}

In [7]:
#todo: check if the double article length really makes sense oh god i cry
conllu_file_stats(KI_corpus)

{'words': 414348,
 'sentences': 15955,
 'av. sentence length': 25.969790034471952,
 'articles': 2329,
 'sentences/article': 6.850579647917561,
 'av. tree height': 5.0140394860545285,
 'max tree height': 13,
 'tree height / sent length': 0.20780959894090187}